In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV

import xgboost as xgb

import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("../data/processed/features_v2.csv")

df['date'] = pd.to_datetime(df['date'])

df = df.sort_values('date')

In [3]:
split_date = df['date'].quantile(0.8)

train = df[df['date'] < split_date]
test  = df[df['date'] >= split_date]

In [4]:
target_finish = "positionOrder"
target_lap    = "avgLapTime_s"
target_points = "points"

drop_cols = [
    "raceId", "driverId", "constructorId",
    "date"
]

features = [col for col in df.columns 
            if col not in drop_cols + [target_finish, target_lap, target_points]]


In [5]:
X_train = train[features]
X_test  = test[features]

y_train = train[target_points]
y_test  = test[target_points]

In [6]:
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

RandomForestRegressor(max_depth=10, min_samples_split=5, n_estimators=300,
                      n_jobs=-1, random_state=42)

In [7]:
rf_preds = rf.predict(X_test)

rf_mae  = mean_absolute_error(y_test, rf_preds)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
rf_r2   = r2_score(y_test, rf_preds)

print("Random Forest Results")
print("MAE:", rf_mae)
print("RMSE:", rf_rmse)
print("R2:", rf_r2)

Random Forest Results
MAE: 3.171435983723194
RMSE: 4.681063320099506
R2: 0.5873552739997292


In [8]:
xgb_model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective="reg:squarederror"
)

xgb_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.05, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=300, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

In [9]:
xgb_preds = xgb_model.predict(X_test)

xgb_mae  = mean_absolute_error(y_test, xgb_preds)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))
xgb_r2   = r2_score(y_test, xgb_preds)

print("XGBoost Results")
print("MAE:", xgb_mae)
print("RMSE:", xgb_rmse)
print("R2:", xgb_r2)

XGBoost Results
MAE: 3.123837611550205
RMSE: 4.724659095068581
R2: 0.5796333801557543


In [11]:
results = pd.DataFrame({
    "model": ["RandomForest_v2", "XGBoost_v2"],
    "target": ["Constructor Points", "Constructor Points"],
    "MAE": [rf_mae, xgb_mae],
    "RMSE": [rf_rmse, xgb_rmse],
    "R2": [rf_r2, xgb_r2]
})

results

,model,target,MAE,RMSE,R2
0,RandomForest_v2,Constructor Points,3.171436,4.681063,0.587355
1,XGBoost_v2,Constructor Points,3.123838,4.724659,0.579633
